# SecureFlow Analytics - Cohort Analysis
## Phase 2: Retention Curves & Cohort Comparisons

**Goal:** Understand user retention patterns and identify high-value cohorts.

**Key Questions:**
1. What's the typical retention curve?
2. Which cohorts retain better (Q4 vs others)?
3. Does feature adoption improve retention?
4. Which channels bring highest LTV users?

---

In [ ]:
import sys
sys.path.append('../src')

import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Custom modules
import config
from analytics.cohort_analysis import (
    calculate_retention_cohorts,
    pivot_retention_table,
    compare_feature_adoption_retention,
    calculate_ltv_by_cohort,
    compare_channel_cohorts
)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 7)

print("✅ Imports successful")

In [ ]:
# Connect to DuckDB
con = duckdb.connect()
print(f"📊 DuckDB connection established")

---
## 1. Weekly Retention Cohorts

Classic cohort analysis: how many users return each week?

In [ ]:
# Calculate retention cohorts
retention_df = calculate_retention_cohorts(con, cohort_period='week', max_periods=8)
print("\n📊 Retention Data (first 20 rows):")
display(retention_df.head(20))

In [ ]:
# Pivot into classic cohort table
retention_table = pivot_retention_table(retention_df)
print("\n📋 Weekly Retention Table:")
display(retention_table)

In [ ]:
# Heatmap visualization
fig, ax = plt.subplots(figsize=(14, 8))

# Format cohort dates
retention_table_display = retention_table.copy()
retention_table_display.index = pd.to_datetime(retention_table_display.index).strftime('%Y-%m-%d')

# Select only retention columns (exclude Cohort Size)
retention_cols = [col for col in retention_table_display.columns if col.startswith('Week')]

sns.heatmap(
    retention_table_display[retention_cols],
    annot=True,
    fmt='.1f',
    cmap='RdYlGn',
    cbar_kws={'label': 'Retention %'},
    vmin=0,
    vmax=100,
    ax=ax
)

ax.set_title('Weekly Retention Cohorts', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Weeks Since Sign Up', fontsize=12)
ax.set_ylabel('Cohort (Sign Up Week)', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# Plot retention curves for recent cohorts
fig, ax = plt.subplots(figsize=(12, 6))

# Get last 5 cohorts
recent_cohorts = retention_df['cohort_period'].unique()[-5:]

for cohort in recent_cohorts:
    cohort_data = retention_df[retention_df['cohort_period'] == cohort]
    ax.plot(
        cohort_data['periods_out'],
        cohort_data['retention_pct'],
        marker='o',
        label=pd.to_datetime(cohort).strftime('%Y-%m-%d'),
        linewidth=2
    )

ax.set_xlabel('Weeks Since Sign Up', fontsize=12)
ax.set_ylabel('Retention %', fontsize=12)
ax.set_title('Retention Curves - Recent Cohorts', fontsize=14, fontweight='bold')
ax.legend(title='Cohort', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## 2. Feature Adoption Impact on Retention

**Hypothesis:** Users who enable real_time_protection have ~40% lower churn.

Let's test it!

In [ ]:
# Compare retention: adopters vs non-adopters
feature_retention = compare_feature_adoption_retention(
    con,
    feature_name='real_time_protection',
    cohort_period='week',
    max_periods=8
)

print("\n🔒 real_time_protection Impact on Retention:")
display(feature_retention.head(20))

In [ ]:
# Plot comparison
fig, ax = plt.subplots(figsize=(12, 6))

# Aggregate across all cohorts
adopted = feature_retention[feature_retention['adoption_status'] == 'Adopted'].groupby('periods_out')['retention_pct'].mean()
not_adopted = feature_retention[feature_retention['adoption_status'] == 'Not Adopted'].groupby('periods_out')['retention_pct'].mean()

ax.plot(adopted.index, adopted.values, marker='o', linewidth=3, label='Adopted real_time_protection', color='#2ca02c')
ax.plot(not_adopted.index, not_adopted.values, marker='o', linewidth=3, label='Did Not Adopt', color='#d62728')

ax.set_xlabel('Weeks Since Sign Up', fontsize=12)
ax.set_ylabel('Average Retention %', fontsize=12)
ax.set_title('Retention: Feature Adopters vs Non-Adopters', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Calculate churn reduction
week_8_adopted = adopted.iloc[-1]
week_8_not_adopted = not_adopted.iloc[-1]
churn_reduction = ((week_8_adopted - week_8_not_adopted) / week_8_not_adopted * 100)

print(f"\n📊 Week 8 Retention:")
print(f"  Adopted: {week_8_adopted:.1f}%")
print(f"  Not Adopted: {week_8_not_adopted:.1f}%")
print(f"  Churn Reduction: {churn_reduction:.1f}%")

---
## 3. Q4 Cohorts vs Others

**Hypothesis:** Q4 cohorts (Oct-Dec) have better retention.

In [ ]:
# Compare Q4 vs non-Q4 cohorts
retention_df['cohort_date'] = pd.to_datetime(retention_df['cohort_period'])
retention_df['is_q4'] = retention_df['cohort_date'].dt.month.isin([10, 11, 12])

q4_retention = retention_df[retention_df['is_q4']].groupby('periods_out')['retention_pct'].mean()
non_q4_retention = retention_df[~retention_df['is_q4']].groupby('periods_out')['retention_pct'].mean()

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(q4_retention.index, q4_retention.values, marker='o', linewidth=3, label='Q4 Cohorts', color='#ff7f0e')
ax.plot(non_q4_retention.index, non_q4_retention.values, marker='o', linewidth=3, label='Non-Q4 Cohorts', color='#1f77b4')

ax.set_xlabel('Weeks Since Sign Up', fontsize=12)
ax.set_ylabel('Average Retention %', fontsize=12)
ax.set_title('Retention: Q4 vs Non-Q4 Cohorts', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n📊 Week 8 Retention:")
print(f"  Q4 Cohorts: {q4_retention.iloc[-1]:.1f}%")
print(f"  Non-Q4 Cohorts: {non_q4_retention.iloc[-1]:.1f}%")
print(f"  Difference: {(q4_retention.iloc[-1] - non_q4_retention.iloc[-1]):.1f} percentage points")

---
## 4. Channel Comparison

Which acquisition channel brings the stickiest users?

In [ ]:
# Compare retention by channel
channel_retention = compare_channel_cohorts(con, cohort_period='week', max_periods=8)

print("\n📱 Retention by Acquisition Channel:")
display(channel_retention.head(20))

In [ ]:
# Plot retention curves by channel
fig, ax = plt.subplots(figsize=(12, 6))

for channel in channel_retention['acquisition_channel'].unique():
    channel_data = channel_retention[channel_retention['acquisition_channel'] == channel]
    avg_retention = channel_data.groupby('weeks_out')['retention_pct'].mean()
    
    ax.plot(
        avg_retention.index,
        avg_retention.values,
        marker='o',
        linewidth=2.5,
        label=channel
    )

ax.set_xlabel('Weeks Since Sign Up', fontsize=12)
ax.set_ylabel('Average Retention %', fontsize=12)
ax.set_title('Retention by Acquisition Channel', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## 5. Lifetime Value (LTV) Analysis

Which cohorts generate the most revenue?

In [ ]:
# Calculate LTV by cohort
ltv_df = calculate_ltv_by_cohort(con, cohort_period='month')

print("\n💰 LTV by Monthly Cohort:")
display(ltv_df)

In [ ]:
# Visualize LTV trend
fig, ax = plt.subplots(figsize=(12, 6))

ltv_df['cohort_date'] = pd.to_datetime(ltv_df['cohort_period'])
ltv_df = ltv_df.sort_values('cohort_date')

ax.plot(
    ltv_df['cohort_date'],
    ltv_df['avg_ltv'],
    marker='o',
    linewidth=2.5,
    color='#2ca02c',
    label='Average LTV'
)

ax.fill_between(
    ltv_df['cohort_date'],
    0,
    ltv_df['avg_ltv'],
    alpha=0.3,
    color='#2ca02c'
)

ax.set_xlabel('Cohort Month', fontsize=12)
ax.set_ylabel('Average LTV ($)', fontsize=12)
ax.set_title('Average Lifetime Value by Cohort', fontsize=14, fontweight='bold')
ax.grid(alpha=0.3)

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# LTV by channel
query = """
WITH user_cohorts AS (
    SELECT 
        user_id,
        acquisition_channel
    FROM read_parquet('G:/My Drive/SecureFlow_Analytics/data/raw/users.parquet')
),
subscription_revenue AS (
    SELECT 
        s.user_id,
        COALESCE(SUM(
            CASE s.subscription_tier
                WHEN 'free' THEN 0
                WHEN 'basic' THEN 4.99
                WHEN 'premium' THEN 9.99
                WHEN 'family' THEN 14.99
            END * DATEDIFF('month', 
                CAST(s.start_date AS TIMESTAMP), 
                COALESCE(CAST(s.end_date AS TIMESTAMP), CURRENT_DATE)
            )
        ), 0) as total_revenue
    FROM read_parquet('G:/My Drive/SecureFlow_Analytics/data/raw/subscriptions.parquet') s
    GROUP BY s.user_id
)
SELECT 
    uc.acquisition_channel,
    COUNT(DISTINCT uc.user_id) as total_users,
    ROUND(AVG(COALESCE(sr.total_revenue, 0)), 2) as avg_ltv,
    ROUND(SUM(COALESCE(sr.total_revenue, 0)), 2) as total_revenue
FROM user_cohorts uc
LEFT JOIN subscription_revenue sr ON uc.user_id = sr.user_id
GROUP BY uc.acquisition_channel
ORDER BY avg_ltv DESC
"""

ltv_by_channel = con.execute(query).df()

print("\n💰 LTV by Acquisition Channel:")
display(ltv_by_channel)

In [ ]:
# Visualize channel LTV
fig, ax = plt.subplots(figsize=(10, 6))

ax.barh(ltv_by_channel['acquisition_channel'], ltv_by_channel['avg_ltv'])
ax.set_xlabel('Average LTV ($)', fontsize=12)
ax.set_title('Average Lifetime Value by Channel', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

for i, (channel, ltv) in enumerate(zip(ltv_by_channel['acquisition_channel'], ltv_by_channel['avg_ltv'])):
    ax.text(ltv, i, f' ${ltv:.2f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

---
## Key Insights

**Summary of findings:**
1. Typical retention curve shape?
2. Feature adoption impact on churn?
3. Best performing cohorts?
4. Highest LTV channel?
5. Recommendations?

In [ ]:
# Close connection
con.close()
print("\n✅ Analysis complete")